# Grievance text analysis 

In [ ]:
from app.config import load_duckdb, directories
import polars as pl
from lingua import Language, LanguageDetectorBuilder
import re
from html import unescape
import seaborn as sns
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
from typing import Iterable, Sequence, Optional

In [ ]:
# Load data
df = load_duckdb(output_format='polars')

In [ ]:
df.shape

## Cleaning the grievance free-text field

In [ ]:
# Strip out html
df = df.with_columns(
    pl.col("grievance")
      .map_elements(lambda s: unescape(s) if s is not None else None)
      .str.replace_all(r"<[^>]+>", "")
      .str.strip_chars()
      .alias("grievance")
)

In [ ]:
# Add other cleaning steps

## Detect language
The approach here is to detect the non-latin scripts first, and then concentrate language detection efforts on the 
latin script remnants. 

In [ ]:

detector = (
    LanguageDetectorBuilder
    .from_languages(Language.ENGLISH, Language.HINDI)
    .with_minimum_relative_distance(0.2)
    .build()
)

script_re = re.compile(r'[\u0B00-\u0B7F\u0900-\u097F]')  # Odia, Devanagari

def label_english_vs_non_english(
    texts: list[str | None],
    *,
    non_english_script_re: re.Pattern = script_re,
    detector=detector,
) -> list[str | None]:
    labels = [None] * len(texts)
    latin_candidates = []
    for i, t in enumerate(texts):
        if t is None or not str(t).strip():
            continue
        s = str(t)
        if non_english_script_re.search(s):
            labels[i] = 'non_en'
        else:
            latin_candidates.append((i, s))

    if latin_candidates:
        idxs, vals = zip(*latin_candidates)
        english_scores = detector.compute_language_confidence_in_parallel(list(vals), Language.ENGLISH)
        for i, score in zip(idxs, english_scores):
            if score < 0.9:
                labels[i] = 'non_en'
            else:
                labels[i] = 'en'
    return labels


labels = label_english_vs_non_english(df['grievance'].to_list())
df = df.with_columns(pl.Series('grievance_lang', labels))


## Basic summary statistics on grievance text

In [ ]:
# Distribution by language
df["grievance_lang"].value_counts(normalize=True)

In [ ]:
df.filter(pl.col("grievance_lang") == "non_en")["grievance"].to_list()

In [ ]:
# Distribution of grievance length
df = df.with_columns(
    pl.col("grievance")
      .str.count_matches(r"\S+")
      .alias("grievance_word_count")
)
df.select("grievance_word_count").describe()

In [ ]:
# mean grievance length by mode (with counts)
by_mode = (
    df.group_by("mode")
      .agg(
          pl.len().alias("count"),
          pl.col("grievance_word_count").mean().round(1).alias("mean_word_count"),
      )
      .sort("count", descending=True)
)
by_mode

In [ ]:
# mean grievance length by language (english vs non-english)
by_lang = (
    df.group_by("grievance_lang")
      .agg(
          pl.len().alias("count"),
          pl.col("grievance_word_count").mean().round(1).alias("mean_word_count"),
      )
      .sort("count", descending=True)
)
by_lang

In [ ]:
# mean grievance length by document status
by_doc_status = (
    df.with_columns(
        pl.when(pl.col("document_url").is_null() | (pl.col("document_url").str.strip_chars() == ""))
          .then(pl.lit("No document present"))
          .otherwise(pl.lit("Document present"))
          .alias("document_status")
    )
    .group_by("document_status")
    .agg(
        pl.len().alias("count"),
        pl.col("grievance_word_count").mean().round(1).alias("mean_word_count"),
    )
)
by_doc_status

In [ ]:
# Grievance type-token ratio
ttr = df.with_columns(
    pl.col("grievance")
      .str.to_lowercase()
      .str.split(" ")
      .alias("tokens")
)

ttr = ttr.with_columns(
    (pl.col("tokens").list.unique().list.len() / pl.col("tokens").list.len())
      .alias("grievance_ttr")
)


In [ ]:
ttr["grievance_ttr"].describe()

## Petitioners with multiple complaints

In [ ]:
df_multi = (
    df.filter(
        pl.col("petitioner_name").is_not_null() &
        (pl.col("petitioner_name").str.strip_chars() != "") &
        (pl.col("petitioner_name").count().over("petitioner_name") >= 2)
    )
    .sort(["petitioner_name", "created_on"])
)

In [ ]:
df_multi[["ticket_no","petitioner_name","category","subcategory","office","dept","grievance","grievance_lang", "created_on","mode", "status", "resolved_on", "benefitted"]]

## Topic analysis

In [ ]:
ortps_labels = [
    "Caste Certificate",
    "Legal Heir Certificate",
    "Birth and death certificate"
    "Partition of land",
    "Conversion of land",
    "Income certificate",
    "Property transfer"
    "Building plan approval",
    "Permission for addition/alteration",
    "Pipe water connection",
    "Marriage certificate",
    "Permission for mortgage",
    "Conveyance deed",
    "Trade licence",
    "Conversion order of leasehold land",
    "Character/Antecedent verification",
    "Copy of FIR",
    "Employee verification request",
    "Fire safety recommendation/certificate",
    "New power connection",
    "Driving licence",
    "Scholarship",
]
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

label_emb = model.encode(ortps_labels, normalize_embeddings=True, show_progress_bar=True)


In [ ]:
def embed_sentences_chunked(
    sentences: Sequence[str] | Iterable[str],
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    out_path: str = directories.MODELS / "embeddings.f32",
    total_sentences: Optional[int] = None,
    chunk_size: int = 50_000,
    batch_size: int = 256,
    normalize: bool = True,
    dtype: np.dtype = np.float32,
):
    """
    Compute sentence embeddings in chunks and write them incrementally to disk.

    Parameters
    ----------
    sentences : Sequence[str] or Iterable[str]
        Sentences to embed. If Iterable, total_sentences must be provided.
    model_name : str
        SentenceTransformers model name.
    out_path : str
        Output path for memmap file.
    total_sentences : int, optional
        Required if sentences is an iterator/generator.
    chunk_size : int
        Number of sentences per outer chunk (controls RAM and Python overhead).
    batch_size : int
        Batch size passed to model.encode (tune for MPS memory).
    normalize : bool
        Whether to L2-normalize embeddings.
    dtype : np.dtype
        Output dtype (float32 recommended; float16 if disk is tight).

    Returns
    -------
    (path, shape)
        Path to embedding file and its shape.
    """
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model = SentenceTransformer(model_name, device=device)

    # ---- probe embedding dim ----
    dim = model.encode(
        ["__probe__"], convert_to_numpy=True, normalize_embeddings=normalize
    ).shape[1]

    # ---- sentence source handling ----
    if isinstance(sentences, Sequence):
        N = len(sentences)
        get_chunk = lambda i, j: sentences[i:j]
        if total_sentences is not None and total_sentences != N:
            raise ValueError("total_sentences does not match len(sentences)")
    else:
        if total_sentences is None:
            raise ValueError("total_sentences must be provided for iterators")
        N = total_sentences
        it = iter(sentences)

        def get_chunk(_, __):
            out = []
            try:
                for _ in range(chunk_size):
                    out.append(next(it))
            except StopIteration:
                pass
            return out

    # ---- prepare output memmap ----
    emb_mm = np.memmap(
        out_path, dtype=dtype, mode="w+", shape=(N, dim)
    )

    # ---- main loop ----
    write_pos = 0
    for start in range(0, N, chunk_size):
        chunk = get_chunk(start, start + chunk_size)
        if not chunk:
            break

        emb = model.encode(
            chunk,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=normalize,
            show_progress_bar=True,
        ).astype(dtype)

        end = write_pos + emb.shape[0]
        emb_mm[write_pos:end] = emb
        emb_mm.flush()
        write_pos = end

    if write_pos != N:
        print(f"Warning: wrote {write_pos} embeddings, expected {N}")

    return out_path, (N, dim)

In [ ]:
# Construct sample for embedding english language grievances
df = df.with_columns(pl.col("grievance").fill_null("").alias("text"))
df_en = df.filter(pl.col("grievance_lang") == "en").sort("id")
df_sample_en = df_en.sample(n=100_000, seed=42, shuffle=True).sort("id")
complaints = df_sample_en["grievance"].to_list()





In [ ]:
df_sample_en["subcategory"].value_counts(sort=True)

In [ ]:
# Embed grievances
path, (N, dim) = embed_sentences_chunked(complaints, out_path = directories.MODELS / "grievance_embeddings.f32")


In [ ]:
path = directories.MODELS / "grievance_embeddings.f32"
compl_emb = np.memmap(path, dtype="float32", mode="r", shape=(N, dim))

In [ ]:
# cosine similarity via dot product
scores = np.matmul(compl_emb, label_emb.T)
best_idx = scores.argmax(axis=1)
best_label = [labels[i] for i in best_idx]
best_score = scores.max(axis=1)

df_sample_en = df_sample_en.with_columns(
    pl.Series("topic_label", best_label),
    pl.Series("topic_score", best_score),
)

### Embedding summary stats

In [ ]:
# Create service related flag
df_sample_en = df_sample_en.with_columns(
    (pl.col("topic_score") > 0.75).alias("service_related")
)

In [ ]:

df_sample_en[["grievance", "topic_label", "topic_score","service_related"]].sort("topic_score", descending=True).head(100).write_excel(workbook = directories.LOGS / "sample_topic_scores.xlsx")

In [ ]:
# Value counts
df_sample_en["service_related"].value_counts()

In [ ]:
# Distribution of topic scores
df_sample_en["topic_score"].describe()

In [ ]:
# By department
by_dept = (
    df_sample_en.group_by("dept")
      .agg(
          pl.len().alias("count"),
          pl.col("topic_score").mean().round(3).alias("avg_topic_score"),
          pl.col("service_related").mean().round(3).alias("share_service_related")
      )
      .sort("avg_topic_score", descending=True)
)
with pl.Config(tbl_rows=41):
    print(by_dept)

In [ ]:
# By subcategory
by_cat = (
    df_sample_en.group_by(["category","subcategory"])
      .agg(
          pl.len().alias("count"),
          pl.col("topic_score").mean().round(3).alias("avg_topic_score"),
      )
      .sort("avg_topic_score", descending=True)
)
with pl.Config(tbl_rows=300):
    print(by_cat)

In [ ]:
# By mode
by_mode = (
    df_sample_en.group_by("mode")
      .agg(
          pl.len().alias("count"),
          pl.col("topic_score").mean().round(3).alias("avg_topic_score"),
      )
      .sort("avg_topic_score", descending=True)
)
by_mode

In [ ]:
# Compare grievance word count vs topic score
p99 = df_sample_en.select(pl.col("grievance_word_count").quantile(0.99)).item()
df_sample_en = df_sample_en.filter(pl.col("grievance_word_count") <= p99)

plot_df = df_sample_en.select(["grievance_word_count", "topic_score"]).drop_nulls()

sns.scatterplot(
    data=plot_df.to_pandas(),
    x="grievance_word_count",
    y="topic_score",
    alpha=0.3
)
plt.xlabel("Grievance word count")
plt.ylabel("Topic score")
plt.title("Topic score vs grievance length")
plt.show()

In [ ]:
# Does grievance text tell us more than cat + subcat combination? 
# Look at cat + subcat ~ ORTPS vs cat + subcat + grievance_text ~ ORTPS
# Want to check if rural housing is dominated by scheme related vs other complaints
# Word cloud for social welfare (category), PMAY (subcategory) and rural housing (sub-category)
# Word cloud for police cases (category) and police cases (sub-category)
# Word cloud for infrastructure (category) and road (sub-category)
# Land

In [ ]:
def wordcloud_from_polars(df, column="grievance", 
                          custom_stopwords=["odisha", "sir", "request", "kindly", "regarding", "please", "give", "madam", "need", "take"]):
    """
    Minimal word cloud from a Polars DataFrame column.
    """
    import re
    from collections import Counter

    import polars as pl
    import matplotlib.pyplot as plt
    from wordcloud import WordCloud, STOPWORDS

    # Pull text column (works for DataFrame or LazyFrame)
    if isinstance(df, pl.LazyFrame):
        texts = df.select(column).collect().get_column(column).to_list()
    else:
        texts = df.get_column(column).to_list()

    # Join corpus
    text = " ".join(t for t in texts if isinstance(t, str)).lower()

    # Light normalization
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-z\s]+", " ", text)

    stopwords = set(STOPWORDS).union(set(custom_stopwords))

    words = [
        w for w in text.split()
        if len(w) > 2 and w not in stopwords
    ]

    freqs = Counter(words)

    wc = WordCloud(
        width=1600,
        height=900,
        background_color="white",
        collocations=False,
    ).generate_from_frequencies(freqs)

    plt.figure(figsize=(10, 6))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.show()

    return wc


In [ ]:
wordcloud_from_polars(df_en, "grievance")

In [ ]:
# Word cloud for housing -- rural housing
df_en_rural_housing = df_en.filter(pl.col("subcategory") == "Rural Housing")
wordcloud_from_polars(df_en_rural_housing)

In [ ]:
df_en_rural_housing["office"].value_counts(normalize=True)

In [ ]:
# Word cloud for PMAY etc under social welfare
df_en_housing_schemes = df_en.filter(pl.col("subcategory") == "IAY/MKY/BPGY/PMAY")
wordcloud_from_polars(df_en_housing_schemes)



In [ ]:
df_en_housing_schemes["office"].value_counts(normalize=True)

In [ ]:
df_en_social_welfare = df_en.filter(pl.col("category") == "Social Welfare")
df_en_social_welfare["office"].value_counts(normalize=True)

In [ ]:
# Social welfare without housing
wordcloud_from_polars(df_en_social_welfare.filter(pl.col("subcategory") != "IAY/MKY/BPGY/PMAY"))

In [ ]:
# Social welfare for ration carfd
wordcloud_from_polars(df_en_social_welfare.filter(pl.col("subcategory") == "Ration Card Issue"))

In [ ]:
df_en_social_welfare.filter(pl.col("subcategory") == "Ration Card Issue")["grievance_word_count"].describe()

# Word lengths time 

In [ ]:
df_en_social_welfare.filter((pl.col("subcategory") == "Ration Card Issue") & (pl.col("mode") == "Website"))["grievance_word_count"].describe()


In [ ]:
# Social welfare for ration card for website mode
wordcloud_from_polars(df_en_social_welfare.filter((pl.col("subcategory") == "Ration Card Issue") & (pl.col("mode") == "Website")))

In [ ]:
# PMAY subcategory complaints for Website
wordcloud_from_polars(df_en_housing_schemes.filter(pl.col("mode") == "Website"))

# Sentence transformers only makes sense for logner complaints.


In [ ]:
# Word cloud for police cases (subcategory)
stopwords = ["odisha", "sir", "request", "kindly", "regarding", "please", "give", "madam", "need", "per", "copy", "respected"]
df_en_subcat_police = df_en.filter(pl.col("subcategory") == "Police Cases")
wordcloud_from_polars(df_en_subcat_police)

In [ ]:
# Word cloud for police case (category) excl police cases
stopwords = ["odisha", "sir", "request", "kindly", "regarding", "please", "give", "madam", "need", "per", "copy", "respected"]
df_en_subcat_police = df_en.filter((pl.col("category") == "Police Case") & (pl.col("subcategory") != "Police Cases"))
wordcloud_from_polars(df_en_subcat_police)

In [ ]:
df_en_infrastructure = df_en.filter(pl.col("category") == "Infrastructure")
wordcloud_from_polars(df_en_infrastructure)

# 2023-24 word count for temple

In [ ]:
wordcloud_from_polars(df_en_infrastructure.filter(pl.col("subcategory") == "Road"))

In [ ]:
# Service matters
df_en_service_matters = df_en.filter(pl.col("category") == "Service Matters")
wordcloud_from_polars(df_en_service_matters)

In [ ]:
# Check -- For those that picked social welfare, check mode
# Do manual reading + sentence transformer 
# Bring in outside information

# == Check within Social welfare == 
# Piped water
# Caste certificate, income certificate
# Check category general subcategory certificates
# Count of complaints where certificate shows up (General cat, certificates subcat) + word count
# Investigate water supply (cat)
# Transport (cat) (check for driving license) 

# ==== ORTPS ====
# Create shares / counts
# Create word counts x do 2024-2025
# For categories / subcategories
#   Social welfare
#       - Ration card
#       - Aadhaar
#   Water Supply
#       - Piped water (A)
#   Infrastructure 
#       - Road
#   Transport
#       - Vehicular transport
#   Education
#       - Scholarships
#   School & College
#       - Scholarships
# 
# ====== Why does categorization matter ==== 
# 
# Central tenets
# - Rural housing
# - Action history / resolution times
# - ORTPS
# - Categorization issue 

# Use last two years

In [ ]:
# Compare website vs non-website modes for both rural housing and PMAY subcats for word length
# Why was there a spike in housing complaints? Did it affect both PMAY vs Rural housing subcategory


In [ ]:
# NaN subcategory check grievance word cloud

In [ ]:
# Subset data created after July 2023
df_en_after_2023 = df_en.filter(pl.col("created_on") >= pl.date(2023, 7, 1))

In [ ]:
def inspect_sample_complaints(df: pl.DataFrame, n: int = 100, word_count_at_least: int = 10):
    sample = df.filter(pl.col("grievance_word_count") > word_count_at_least).sample(n=n, seed=45)
    return sample["grievance"].to_list()

In [ ]:
# Number of complaints by year 
# Fiscal year labeled by the July start year (e.g., 2023 covers 2023-07-01..2024-06-30)
result = (
    df_en_after_2023.with_columns(
        pl.when(pl.col("created_on").dt.month() >= 7)
        .then(pl.col("created_on").dt.year())
        .otherwise(pl.col("created_on").dt.year() - 1)
        .alias("july_year")
    )
    .group_by("july_year")
    .len()
    .sort("july_year")
)

result

## Investigating ORTPS services

## Set up embedding production

In [ ]:
from pathlib import Path
from typing import Iterable, Sequence, Optional

import numpy as np
import polars as pl
import torch
from sentence_transformers import SentenceTransformer


def _embeddings_file_matches(out_path: str, *, N: int, dim: int, dtype: np.dtype) -> bool:
    out_path = Path(out_path)
    if not out_path.exists():
        return False
    expected_bytes = int(N) * int(dim) * np.dtype(dtype).itemsize
    try:
        return out_path.stat().st_size == expected_bytes
    except OSError:
        return False


def get_or_create_embeddings(
    sentences: Sequence[str] | Iterable[str],
    *,
    total_sentences: int,
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    out_path: str = directories.MODELS / "embeddings.f32",
    chunk_size: int = 50_000,
    batch_size: int = 256,
    normalize: bool = True,
    dtype: np.dtype = np.float32,
):
    """
    Load embeddings from disk if shape matches; otherwise recompute.
    Returns (embeddings_memmap, (N, dim)).
    """
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model = SentenceTransformer(model_name, device=device)

    dim = model.encode(
        ["__probe__"], convert_to_numpy=True, normalize_embeddings=normalize
    ).shape[1]

    if not _embeddings_file_matches(out_path, N=total_sentences, dim=dim, dtype=dtype):
        embed_sentences_chunked(
            sentences,
            model_name=model_name,
            out_path=out_path,
            total_sentences=total_sentences,
            chunk_size=chunk_size,
            batch_size=batch_size,
            normalize=normalize,
            dtype=dtype,
        )

    emb = np.memmap(out_path, dtype=dtype, mode="r", shape=(total_sentences, dim))
    return emb, (total_sentences, dim)


def cosine_similarity_scores(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    If embeddings are normalized, cosine similarity == dot product.
    Returns matrix shape (len(a), len(b)).
    """
    return a @ b.T


def assign_grievance_labels(
    df: pl.DataFrame,
    labels: Sequence[str],
    *,
    text_col: str = "grievance",
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    out_path: str = directories.MODELS / "embeddings.f32",
    chunk_size: int = 50_000,
    embed_batch_size: int = 256,
    sim_batch_size: int = 2048,
    normalize: bool = True,
    dtype: np.dtype = np.float32,
) -> pl.DataFrame:
    """
    Ensure grievance embeddings exist and match df size, compute label embeddings,
    and assign top-1 label + score per row.
    """
    if text_col not in df.columns:
        raise KeyError(f"Missing column: {text_col}")

    texts = df.get_column(text_col).fill_null("").cast(pl.Utf8).to_list()
    N = len(texts)

    emb, _shape = get_or_create_embeddings(
        texts,
        total_sentences=N,
        model_name=model_name,
        out_path=out_path,
        chunk_size=chunk_size,
        batch_size=embed_batch_size,
        normalize=normalize,
        dtype=dtype,
    )

    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model = SentenceTransformer(model_name, device=device)
    label_emb = model.encode(
        list(labels),
        convert_to_numpy=True,
        normalize_embeddings=normalize,
    ).astype(dtype)

    best_idx = np.empty(N, dtype=np.int64)
    best_score = np.empty(N, dtype=np.float32)

    for start in range(0, N, sim_batch_size):
        end = min(start + sim_batch_size, N)
        scores = cosine_similarity_scores(emb[start:end], label_emb)
        idx = scores.argmax(axis=1)
        best_idx[start:end] = idx
        best_score[start:end] = scores[np.arange(end - start), idx]

    return df.with_columns(
        pl.Series("grievance_label", [labels[i] for i in best_idx]),
        pl.Series("grievance_score", best_score),
    )


In [ ]:
def compute_share(main_df: pl.DataFrame, subset_df: pl.DataFrame, subset_name: str):
    share = subset_df.shape[0] / main_df.shape[0]
    print(f"Total obs. overall: {main_df.shape[0]}")
    print(f"Share in {subset_name} = {share:.1%}")

### Ration card issues

In [ ]:
df_en_after_2023_ration_card = df_en_after_2023.filter(pl.col("subcategory") == "Ration Card Issue")
compute_share(df_en_after_2023, df_en_after_2023_ration_card, "Ration Card")

In [ ]:
# Word cloud  ration card
wordcloud_from_polars(df_en_after_2023_ration_card)

In [ ]:
# Inspect sample
inspect_sample_complaints(df_en_after_2023_ration_card, n=20)

In [ ]:
inspect_sample_complaints(df_en_after_2023_ration_card, n=20, word_count_at_least=1)

In [ ]:
df_en_after_2023_ration_card_labelled = assign_grievance_labels(df_en_after_2023_ration_card,
                                                                labels=)

### Aadhaar

In [ ]:
# Share
df_en_after_2023_aadhar = df_en_after_2023.filter(pl.col("subcategory") == "Aadhar Card Issues")
compute_share(df_en_after_2023, df_en_after_2023_aadhar, "Aadhaar Card")

In [ ]:
wordcloud_from_polars(df_en_after_2023_aadhar)

In [ ]:
inspect_sample_complaints(df_en_after_2023_aadhar, n=20, word_count_at_least=1)

In [ ]:
inspect_sample_complaints(df_en_after_2023_aadhar, n=20, word_count_at_least=10)

### Piped water

In [ ]:
df_en_after_2023_piped_water = df_en_after_2023.filter(pl.col("subcategory").str.contains("Piped Water"))
df_en_after_2023_piped_water.shape

In [ ]:
wordcloud_from_polars(df_en_after_2023_piped_water)

In [ ]:
inspect_sample_complaints(df_en_after_2023_piped_water, n=20)

In [ ]:
inspect_sample_complaints(df_en_after_2023_piped_water, n = 20,word_count_at_least= 1)

### Vehicle transportation

In [ ]:
df_en_after_2023_vehicle_transport = df_en_after_2023.filter(pl.col("subcategory") == "Vehicle Transportation")
df_en_after_2023_vehicle_transport.shape

In [ ]:
wordcloud_from_polars(df_en_after_2023_vehicle_transport)

In [ ]:
inspect_sample_complaints(df_en_after_2023_vehicle_transport, n=20)

In [ ]:
# Education scholarships and school and college scholarships
df_en_after_2023_educ = df_en_after_2023.filter((pl.col("category") == "Education") & (pl.col("subcategory").str.contains("Scholarship")))
df_en_after_2023_educ.shape

In [ ]:
wordcloud_from_polars(df_en_after_2023_educ)

In [ ]:
inspect_sample_complaints(df_en_after_2023_educ)

In [ ]:
## Looking at NaN Complaints
df_en_after_2023_nan = df_en_after_2023.filter(pl.col("category").is_null())
compute_share(df_en_after_2023, df_en_after_2023_nan, "Missung category")

In [ ]:
wordcloud_from_polars(df_en_after_2023_nan)

In [ ]:
inspect_sample_complaints(df_en_after_2023_nan, n=20)

In [ ]:
inspect_sample_complaints(df_en_after_2023_nan, n=20, word_count_at_least=1)

In [ ]:
# Looking at certificates
df_en_after_2023_certificates = df_en_after_2023.filter(pl.col("subcategory") == "Certificates")
df_en_after_2023_certificates.shape

In [ ]:
wordcloud_from_polars(df_en_after_2023_certificates)

In [ ]:
inspect_sample_complaints(df_en_after_2023_certificates, n=20)

In [ ]:
inspect_sample_complaints(df_en_after_2023_certificates, n=20, word_count_at_least=1)

# Departments and categories by ORTPS label


In [ ]:
from typing import Sequence
import numpy as np
import polars as pl
import torch
from sentence_transformers import SentenceTransformer


def _factorize(values: list[str]) -> tuple[np.ndarray, list[str]]:
    mapping: dict[str, int] = {}
    uniques: list[str] = []
    codes = np.empty(len(values), dtype=np.int64)
    for i, v in enumerate(values):
        if v not in mapping:
            mapping[v] = len(uniques)
            uniques.append(v)
        codes[i] = mapping[v]
    return codes, uniques


def top_department_category_per_label(
    df: pl.DataFrame,
    labels: Sequence[str],
    *,
    text_col: str = "grievance",
    dept_col: str = "department",
    cat_col: str = "category",
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    out_path: str = directories.MODELS / "embeddings.f32",
    chunk_size: int = 50_000,
    embed_batch_size: int = 256,
    sim_batch_size: int = 4096,
    normalize: bool = True,
    dtype: np.dtype = np.float32,
) -> pl.DataFrame:
    if text_col not in df.columns:
        raise KeyError(f"Missing column: {text_col}")
    if dept_col not in df.columns:
        raise KeyError(f"Missing column: {dept_col}")
    if cat_col not in df.columns:
        raise KeyError(f"Missing column: {cat_col}")

    texts = df.get_column(text_col).fill_null("").cast(pl.Utf8).to_list()
    departments = df.get_column(dept_col).fill_null("").cast(pl.Utf8).to_list()
    categories = df.get_column(cat_col).fill_null("").cast(pl.Utf8).to_list()

    N = len(texts)
    if N == 0:
        return pl.DataFrame(
            {
                "label": [],
                "top_department": [],
                "top_department_avg": [],
                "top_category": [],
                "top_category_avg": [],
            }
        )

    emb, _shape = get_or_create_embeddings(
        texts,
        total_sentences=N,
        model_name=model_name,
        out_path=out_path,
        chunk_size=chunk_size,
        batch_size=embed_batch_size,
        normalize=normalize,
        dtype=dtype,
    )

    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model = SentenceTransformer(model_name, device=device)
    label_emb = model.encode(
        list(labels),
        convert_to_numpy=True,
        normalize_embeddings=normalize,
    ).astype(dtype)

    dept_codes, dept_uniques = _factorize(departments)
    cat_codes, cat_uniques = _factorize(categories)

    dept_counts = np.bincount(dept_codes, minlength=len(dept_uniques)).astype(np.float64)
    cat_counts = np.bincount(cat_codes, minlength=len(cat_uniques)).astype(np.float64)

    L = len(labels)
    dept_sums = np.zeros((L, len(dept_uniques)), dtype=np.float64)
    cat_sums = np.zeros((L, len(cat_uniques)), dtype=np.float64)

    for start in range(0, N, sim_batch_size):
        end = min(start + sim_batch_size, N)
        scores = cosine_similarity_scores(emb[start:end], label_emb)  # (b, L)
        dept_batch = dept_codes[start:end]
        cat_batch = cat_codes[start:end]

        for j in range(L):
            dept_sums[j] += np.bincount(
                dept_batch, weights=scores[:, j], minlength=len(dept_uniques)
            )
            cat_sums[j] += np.bincount(
                cat_batch, weights=scores[:, j], minlength=len(cat_uniques)
            )

    dept_avg = dept_sums / dept_counts[None, :]
    cat_avg = cat_sums / cat_counts[None, :]

    top_dept_idx = dept_avg.argmax(axis=1)
    top_cat_idx = cat_avg.argmax(axis=1)

    return pl.DataFrame(
        {
            "label": list(labels),
            "top_department": [dept_uniques[i] for i in top_dept_idx],
            "top_department_avg": dept_avg[np.arange(L), top_dept_idx],
            "top_category": [cat_uniques[i] for i in top_cat_idx],
            "top_category_avg": cat_avg[np.arange(L), top_cat_idx],
        }
    )


In [ ]:
with pl.Config(tbl_rows=1000):
    print(top_department_category_per_label(df_sample_en,labels=ortps_labels, dept_col="dept", cat_col="category",
                                  model_name="sentence-transformers/all-MiniLM-L6-v2", out_path= directories.MODELS / "grievance_embeddings.f32"))

## Draft ORTPS output (pass to separate notebook)


In [ ]:
from __future__ import annotations

from typing import Sequence, Optional
import re
import polars as pl
import pandas as pd


def phrase_counts_by_year_wide(
    df: pl.DataFrame,
    phrases: Sequence[str],
    *,
    text_col: str = "grievance",
    year_col: Optional[str] = None,
    date_col: Optional[str] = "date",
    case_sensitive: bool = False,
    literal: bool = True,
) -> pl.DataFrame:
    """
    Wide output with columns:
      year, count_<phrase_key>, share_<phrase_key> (percent, 1 decimal)
    """
    if text_col not in df.columns:
        raise KeyError(f"Missing column: {text_col}")

    if year_col is None:
        if date_col is None or date_col not in df.columns:
            raise KeyError("Provide year_col or a valid date_col")
        year_expr = pl.col(date_col).dt.year()
    else:
        if year_col not in df.columns:
            raise KeyError(f"Missing column: {year_col}")
        year_expr = pl.col(year_col)

    if not phrases:
        return pl.DataFrame({"year": []})

    def _contains_expr(phrase: str, *, literal: bool, case_sensitive: bool) -> pl.Expr:
        text = pl.col("_text")
        if not case_sensitive:
            text = text.str.to_lowercase()
            phrase = phrase.lower()
        return text.str.contains(phrase, literal=literal)

    def _slug(s: str) -> str:
        s = s.strip().lower()
        s = re.sub(r"[^0-9a-zA-Z]+", "_", s)
        s = re.sub(r"_+", "_", s).strip("_")
        return s or "phrase"

    # build unique keys
    keys = []
    seen = {}
    for p in phrases:
        k = _slug(p)
        if k in seen:
            seen[k] += 1
            k = f"{k}_{seen[k]}"
        else:
            seen[k] = 0
        keys.append(k)

    base = df.with_columns(
        _year=year_expr,
        _text=pl.col(text_col).fill_null("").cast(pl.Utf8),
    )

    flag_cols = []
    for i, (phrase, key) in enumerate(zip(phrases, keys)):
        flag = f"_has_{i}"
        base = base.with_columns(
            _contains_expr(phrase, literal=literal, case_sensitive=case_sensitive).alias(flag)
        )
        flag_cols.append((flag, key))

    totals = base.group_by("_year").agg(total=pl.len())

    counts = base.group_by("_year").agg(
        [pl.col(flag).sum().alias(f"count_{key}") for flag, key in flag_cols]
    )

    share_exprs = [
        (pl.col(f"count_{key}") / pl.col("total") * 100.0).round(1).alias(f"share_{key}")
        for _, key in flag_cols
    ]

    return (
        counts.join(totals, on="_year", how="left")
        .with_columns(share_exprs)
        .select(
            pl.col("_year").alias("year"),
            *[f"count_{key}" for _, key in flag_cols],
            *[f"share_{key}" for _, key in flag_cols],
        )
        .sort("year")
    )

import xlsxwriter
import numpy as np

def export_phrase_counts_excel(
    wide_df: pl.DataFrame,
    phrases: Sequence[str],
    *,
    path: str,
    sheet_name: str = "phrase_counts",
) -> None:
    def _slug(s: str) -> str:
        s = s.strip().lower()
        s = re.sub(r"[^0-9a-zA-Z]+", "_", s)
        s = re.sub(r"_+", "_", s).strip("_")
        return s or "phrase"

    # build unique keys (match wide output)
    keys = []
    seen = {}
    for p in phrases:
        k = _slug(p)
        if k in seen:
            seen[k] += 1
            k = f"{k}_{seen[k]}"
        else:
            seen[k] = 0
        keys.append(k)

    ordered_cols = ["year"]
    for k in keys:
        ordered_cols.append(f"count_{k}")
        ordered_cols.append(f"share_{k}")

    data = wide_df.select(ordered_cols).to_numpy()

    workbook = xlsxwriter.Workbook(path)
    ws = workbook.add_worksheet(sheet_name)

    header_fmt = workbook.add_format({"bold": True, "align": "center", "valign": "vcenter"})
    count_fmt = workbook.add_format({"num_format": "0"})
    share_fmt = workbook.add_format({"num_format": '0.0"%"'})
    year_fmt = workbook.add_format({"num_format": "0"})

    # headers
    ws.merge_range(0, 0, 1, 0, "Year", header_fmt)
    for i, phrase in enumerate(phrases):
        c = 1 + 2 * i
        ws.merge_range(0, c, 0, c + 1, phrase, header_fmt)
        ws.write(1, c, "Count", header_fmt)
        ws.write(1, c + 1, "% Share", header_fmt)

    # data
    row_start = 2
    for r, row in enumerate(data, start=row_start):
        ws.write_number(r, 0, float(row[0]), year_fmt)
        for i in range(len(phrases)):
            c = 1 + 2 * i
            count_val = row[c]
            share_val = row[c + 1]
            if count_val is not None and not (isinstance(count_val, float) and np.isnan(count_val)):
                ws.write_number(r, c, float(count_val), count_fmt)
            else:
                ws.write_blank(r, c, None)
            if share_val is not None and not (isinstance(share_val, float) and np.isnan(share_val)):
                ws.write_number(r, c + 1, float(share_val), share_fmt)
            else:
                ws.write_blank(r, c + 1, None)

    # column widths
    ws.set_column(0, 0, 10)
    for i in range(len(phrases)):
        ws.set_column(1 + 2 * i, 1 + 2 * i + 1, 14)

    workbook.close()

In [ ]:
ortps_labels = ["Caste certificate", "Income certificate", "Scholarship", "Ration card"]

In [ ]:

count_shares_by_year = phrase_counts_by_year_wide(df_en_after_2023, phrases=ortps_labels, date_col="created_on")

In [ ]:
export_phrase_counts_excel(wide_df=count_shares_by_year, phrases=ortps_labels, path=directories.LOGS / "phrase_counts.xlsx")

In [ ]:
# Word clouds without key var